# FineWeb-Edu Llama 124M: smoke test, probe and learning-rate sweep

This notebook prepares the full 124M training run of [Track 2](../../README.md#2-practicing-at-124m-a-llama-style-model-on-fineweb-edu). It builds the FineWeb-Edu token cache once, checks the model and training loop, measures the GPU, and sweeps the peak learning rate with the exact setup of the final Modal H100 run.

| Step | Runtime | What it does |
| --- | --- | --- |
| 1. Token cache | **CPU** (first time only) | Downloads FineWeb-Edu `sample/10BT` shards, trains a 32,000-entry BPE tokenizer, tokenizes about 1.5B training and 10M validation tokens, and saves everything to Drive |
| 2. Smoke test | A100 or L4 | Overfits one batch, checks that loss falls without NaNs, and checks that a Drive checkpoint resumes to the identical loss |
| 3. Probe | A100 or L4 | Measures tokens per second and peak memory for each micro-batch and picks the fastest one that fits |
| 4. LR sweep | A100 or L4 | Trains the 124,668,672-parameter model at each peak learning rate and compares validation loss |
| 5. Results | any | Table, curves, recommended learning rate, and files to commit to the repo |

### How to run it

1. **First time:** choose **Runtime → Change runtime type → CPU** (High-RAM if offered) and select **Runtime → Run all**. The notebook builds the token cache, saves it to Drive, and stops. This takes about 1–2 hours and uses no GPU time.
2. **Then:** switch to an **A100** (or an **L4** if no A100 is available) and select **Run all** again. Approve the two Google sign-in prompts: one mounts Drive, and the other lets the notebook permanently delete its own old checkpoints from the Drive trash.
3. **After a disconnect:** reconnect and select **Run all** with the same `EXPERIMENT_NAME`. Finished learning rates are skipped and the active one resumes from its last hourly checkpoint. Google AI Pro doesn't run notebooks in the background, so keep the browser tab open.

| Plan (set by the GPU the sweep starts on) | Learning rates | Steps per rate | Tokens per rate | Expected time |
| --- | --- | ---: | ---: | ---: |
| A100 | `3e-4, 6e-4, 1e-3, 2e-3, 3e-3` | 600 | 315M | ~3–3.5 h |
| L4 | `6e-4, 1e-3, 2e-3, 3e-3` | 400 | 210M | ~6 h, over two sessions |

Every rate starts from the same weights and sees the same batches. The L4 plan drops `3e-4`, which is almost surely too low, and keeps 400 steps so that each run spends most of its time after the 250-step warmup. The probe replaces the expected times with measured ones.

**Drive usage.** The cache takes about 3 GB. At most **one** training checkpoint (about 1.5 GB) exists at any time: each hourly save replaces the previous one, and the notebook purges its own replaced files from the Drive trash so they don't keep using quota.

## 0. Settings

Everything you might change lives in this cell. The model, batch and warmup settings must match the final Modal run, so leave them alone unless the plan changes.

In [ ]:
# ---- Run control ----
PROJECT_DIR_NAME = 'CS336_FineWebEdu_Llama124M'   # folder in MyDrive
EXPERIMENT_NAME = 'lr_sweep_v1'                   # reuse the name to resume; change it to start over
RUN_SMOKE_TEST = True          # skipped automatically once it has passed on this GPU type
RUN_PROBE = True               # reuses this GPU type's saved probe unless REPROBE = True
REPROBE = False
RUN_SWEEP = True
MAX_UPDATES_THIS_SESSION = None  # e.g. 5 to test pausing and resuming
REBUILD_CACHE = False          # True downloads and tokenizes again even if a good cache exists
ALLOW_BUILD_ON_GPU = False     # tokenizing is CPU work; build on a CPU runtime
PURGE_DRIVE_TRASH = True       # permanently delete this notebook's own trashed files only

# ---- Data (changing any of these means rebuilding the cache) ----
DATASET = 'HuggingFaceFW/fineweb-edu'
DATASET_SUBSET = 'sample/10BT'
TRAIN_SHARDS = 2               # the subset's first N parquet files, about 0.7-0.8B tokens each
VALID_TOKENS = 10_000_000      # taken from the subset's last file, which training never reads
VOCAB_SIZE = 32_000
EOS_TOKEN = '<|endoftext|>'
TOKENIZER_TRAIN_DOCS = 200_000

# ---- Model and training: identical to the final Modal run ----
CONTEXT = 1024
EFFECTIVE_BATCH = 512          # sequences per optimizer step: 512 x 1,024 = 524,288 tokens
WARMUP_STEPS = 250
WEIGHT_DECAY = 0.1
BETAS = (0.9, 0.95)
GRAD_CLIP = 1.0
SEED = 336

# ---- Sweep plans, chosen by the GPU the experiment starts on ----
SWEEP_PLANS = {
    'a100': {'learning_rates': (3e-4, 6e-4, 1e-3, 2e-3, 3e-3), 'steps': 600, 'microbatches': (64, 32, 16, 8)},
    'l4': {'learning_rates': (6e-4, 1e-3, 2e-3, 3e-3), 'steps': 400, 'microbatches': (16, 8, 4)},
}
EVAL_EVERY = 50                # steps between validation passes
VAL_WINDOWS = 512              # fixed validation windows of 1,024 tokens
LOG_EVERY = 10
CHECKPOINT_EVERY_SECONDS = 3600
PROBE_TIMED_STEPS = 2
PROBE_HEADROOM_GIB = 1.0
CLOSE_MARGIN = 0.01            # nats: prefer the lower rate when final losses are this close
SAMPLE_PROMPT = 'The water cycle begins when'

## 1. Runtime, Drive and GPU profile

The sweep needs bf16, so it runs on an A100 or L4 (or any GPU with compute capability 8.0 or newer). A T4 is refused. A CPU runtime is only used to build the token cache.

In [ ]:
import gc
import hashlib
import itertools
import json
import math
import os
import shutil
import sys
import time
from dataclasses import asdict, dataclass
from pathlib import Path

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'true')

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from google.colab import drive


class StopHere(Exception):
    'Raised on purpose to stop Run all with an explanation.'


drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive') / PROJECT_DIR_NAME
CACHE_DRIVE = DRIVE_ROOT / 'token_cache'
CACHE_LOCAL = Path('/content/token_cache')
RUN_DIR = DRIVE_ROOT / 'runs' / EXPERIMENT_NAME
CKPT_DIR = RUN_DIR / 'checkpoints'
SMOKE_DIR = DRIVE_ROOT / 'smoke_tests'
LOCAL_TMP = Path('/content/ckpt_tmp')
FILE_PREFIX = 'cs336-llama124m-'   # the trash purge only touches files with this prefix
for folder in (DRIVE_ROOT, RUN_DIR, CKPT_DIR, SMOKE_DIR, LOCAL_TMP):
    folder.mkdir(parents=True, exist_ok=True)

PEAK_BF16_FLOPS = {'a100': 312e12, 'h100': 989e12, 'l4': 121e12}


def detect_runtime():
    if not torch.cuda.is_available():
        return torch.device('cpu'), 'cpu', None, 'CPU'
    name = torch.cuda.get_device_name(0)
    major, minor = torch.cuda.get_device_capability(0)
    upper = name.upper()
    if 'A100' in upper:
        tag = 'a100'
    elif 'H100' in upper:
        tag = 'h100'
    elif 'L4' in upper:
        tag = 'l4'
    else:
        tag = 'gpu-' + ''.join(ch if ch.isalnum() else '-' for ch in name.lower()).strip('-')
    if major < 8:
        tag = 'no-bf16'
    plan = 'a100' if tag in ('a100', 'h100') else 'l4'
    return torch.device('cuda'), tag, plan, f'{name} (compute capability {major}.{minor})'


DEVICE, GPU_TAG, GPU_PLAN, GPU_NAME = detect_runtime()
USE_COMPILE = DEVICE.type == 'cuda'
if DEVICE.type == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    free, total = torch.cuda.mem_get_info()
    print(f'{GPU_NAME} | {free / 2**30:.1f} of {total / 2**30:.1f} GiB free')
else:
    print('CPU runtime: this session can build the token cache but cannot train.')
print(f'PyTorch {torch.__version__} | {os.cpu_count()} CPU cores | '
      f'{shutil.disk_usage("/content").free / 2**30:.0f} GiB free on the VM disk')
print(f'Drive folder: {DRIVE_ROOT}')


def require_training_gpu():
    if DEVICE.type != 'cuda':
        raise StopHere('Switch to an A100 or L4 runtime (Runtime → Change runtime type) and Run all again.')
    if GPU_TAG == 'no-bf16':
        raise StopHere(f'{GPU_NAME} has no bf16. Track 2 tests on an A100 or L4; the T4 was used for Track 1 only.')


# Small helpers that also work on CPU, so the code paths stay the same.
def gpu_sync():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()


def gpu_free_bytes():
    return torch.cuda.mem_get_info()[0] if DEVICE.type == 'cuda' else 0


def gpu_reset_peak():
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def gpu_peak_bytes():
    return torch.cuda.max_memory_allocated() if DEVICE.type == 'cuda' else 0


def read_json(path):
    path = Path(path)
    return json.loads(path.read_text(encoding='utf-8')) if path.is_file() else None


def write_json(path, data):
    path = Path(path)
    tmp = path.with_name(path.name + '.partial')
    tmp.write_text(json.dumps(data, indent=2, allow_nan=False) + '\n', encoding='utf-8')
    os.replace(tmp, path)

### Drive trash purge

Files deleted or replaced through Colab's Drive mount can move to the Drive trash, where they still count against your quota. With a 1.5 GB checkpoint every hour, that adds up. After each checkpoint the notebook asks the Drive API for **trashed files whose names start with `cs336-llama124m-`** and deletes those permanently. Nothing else in your trash is touched. Set `PURGE_DRIVE_TRASH = False` to skip this; the smoke test then reports whether the trash is growing.

In [ ]:
class DriveTrash:
    def __init__(self, enabled):
        self.enabled = enabled
        self.service = None

    def connect(self):
        if not self.enabled or self.service is not None:
            return
        from google.colab import auth
        from googleapiclient.discovery import build
        auth.authenticate_user()
        self.service = build('drive', 'v3', cache_discovery=False)

    def quota(self):
        if self.service is None:
            return None
        q = self.service.about().get(fields='storageQuota').execute()['storageQuota']
        return {key: int(q[key]) for key in ('usage', 'usageInDriveTrash') if key in q}

    def purge(self):
        # Permanently deletes this notebook's own files from the trash. Returns the number deleted.
        if self.service is None:
            return 0
        deleted, token = 0, None
        try:
            while True:
                page = self.service.files().list(
                    q="trashed = true and 'me' in owners", spaces='drive',
                    fields='nextPageToken, files(id, name)', pageSize=1000, pageToken=token,
                ).execute()
                for item in page.get('files', []):
                    if item['name'].startswith(FILE_PREFIX):
                        self.service.files().delete(fileId=item['id']).execute()
                        deleted += 1
                token = page.get('nextPageToken')
                if not token:
                    return deleted
        except Exception as error:  # a failed purge never stops training
            print(f'Drive trash purge skipped: {error}')
            return deleted


DRIVE_TRASH = DriveTrash(PURGE_DRIVE_TRASH and DEVICE.type == 'cuda')
DRIVE_TRASH.connect()
if DRIVE_TRASH.service is not None:
    usage = DRIVE_TRASH.quota()
    print(f'Drive API connected | used {usage["usage"] / 2**30:.1f} GiB, '
          f'{usage.get("usageInDriveTrash", 0) / 2**30:.2f} GiB of it in the trash')

## 2. Token cache: load, or build once, then check

Training reads random windows from one flat stream of token IDs. Building that stream is slow (download, tokenizer training, tokenization) but deterministic, so it happens once and is reused:

1. **Cache on the VM disk** (same session) → check it and use it.
2. **Cache on Drive** → copy it to the VM disk (about 3 GB, a few minutes), check it, and use it. Training reads from the local copy because random reads over the Drive mount are slow.
3. **No good cache anywhere** → build it, on a CPU runtime:
   - Pin the dataset to its current commit and download `sample/10BT` parquet shards one at a time.
   - Train a byte-level BPE tokenizer with 32,000 entries (`<|endoftext|>` plus 31,999 learned tokens and bytes) on the first 200,000 documents.
   - Encode the first `TRAIN_SHARDS` files into `train.u16`, appending `<|endoftext|>` after every document, and the first 10M tokens of the **last** file into `valid.u16`.
   - Write `manifest.json` last. A cache without a manifest is treated as incomplete.

**The check** runs on every load, and fails loudly instead of letting training run on bad data:

| Check | Catches |
| --- | --- |
| Manifest fields match the settings (dataset, subset, vocabulary size, EOS token, dtype) | A cache built with different settings |
| SHA-256 of `tokenizer.json` matches the manifest, vocabulary is 32,000, EOS ID matches | A different or edited tokenizer, which would make the token IDs meaningless |
| File size = token count × 2 bytes, and a fingerprint of the first and last 8 MiB matches | Truncated or corrupted copies |
| Sampled token IDs are below 32,000, and the average document length is plausible | Wrong dtype or garbage data |
| Decoded windows from both splits | A quick look at the text with your own eyes |

The Modal run must reuse this exact `tokenizer.json`. Its hash is recorded in the manifest, in every checkpoint and in the sweep summary.

In [ ]:
from tokenizers import Tokenizer


class CacheError(Exception):
    pass


def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()


def fingerprint(path, chunk=8 * 2**20):
    # Size plus the first and last 8 MiB: fast, and catches truncated or mixed-up files.
    size = path.stat().st_size
    digest = hashlib.sha256(str(size).encode())
    with open(path, 'rb') as f:
        digest.update(f.read(chunk))
        if size > chunk:
            f.seek(max(chunk, size - chunk))
            digest.update(f.read(chunk))
    return digest.hexdigest()


def verify_cache(folder, quiet=False):
    folder = Path(folder)
    manifest = read_json(folder / 'manifest.json')
    if manifest is None:
        raise CacheError(f'no manifest.json in {folder}')
    expected = {'format_version': 1, 'dataset': DATASET, 'subset': DATASET_SUBSET,
                'vocab_size': VOCAB_SIZE, 'eos_token': EOS_TOKEN, 'dtype': 'uint16'}
    for key, value in expected.items():
        if manifest.get(key) != value:
            raise CacheError(f'manifest {key} = {manifest.get(key)!r}, expected {value!r}')
    tokenizer_path = folder / 'tokenizer.json'
    if not tokenizer_path.is_file():
        raise CacheError('tokenizer.json is missing')
    text = tokenizer_path.read_text(encoding='utf-8')
    if sha256_bytes(text.encode('utf-8')) != manifest['tokenizer_sha256']:
        raise CacheError('tokenizer.json does not match the SHA-256 in the manifest')
    tokenizer = Tokenizer.from_str(text)
    if tokenizer.get_vocab_size() != VOCAB_SIZE or tokenizer.token_to_id(EOS_TOKEN) != manifest['eos_id']:
        raise CacheError('tokenizer vocabulary or EOS ID does not match the manifest')
    streams = {}
    rng = np.random.default_rng(0)
    for split in ('train', 'valid'):
        info = manifest['splits'][split]
        path = folder / f'{split}.u16'
        if not path.is_file():
            raise CacheError(f'{path.name} is missing')
        if path.stat().st_size != info['tokens'] * 2:
            raise CacheError(f'{path.name} has {path.stat().st_size:,} bytes, expected {info["tokens"] * 2:,}')
        if fingerprint(path) != info['fingerprint']:
            raise CacheError(f'{path.name} fingerprint does not match the manifest')
        stream = np.memmap(path, dtype=np.uint16, mode='r')
        if len(stream) <= CONTEXT + 1:
            raise CacheError(f'{path.name} is shorter than one training window')
        sample = stream[rng.integers(0, len(stream), size=min(1_000_000, len(stream)))]
        if int(sample.max()) >= VOCAB_SIZE:
            raise CacheError(f'{path.name} contains token IDs outside the vocabulary')
        eos_share = float((sample == manifest['eos_id']).mean())
        mean_doc = 1 / eos_share if eos_share else float('inf')
        if not 50 <= mean_doc <= 20_000:
            raise CacheError(f'{path.name}: implausible mean document length of {mean_doc:,.0f} tokens')
        streams[split] = stream
        if not quiet:
            start = int(rng.integers(0, len(stream) - 80))
            snippet = tokenizer.decode(stream[start:start + 80].astype(np.int64).tolist())
            print(f'  {split}: {info["tokens"]:,} tokens, {info["documents"]:,} documents, '
                  f'~{mean_doc:,.0f} tokens/document')
            print(f'    "{snippet[:240].strip()}"')
    if not quiet:
        print(f'  tokenizer SHA-256 {manifest["tokenizer_sha256"][:16]}… | dataset revision '
              f'{manifest["revision"][:12]} | built {manifest["built_utc"]}')
    return manifest, tokenizer, streams


def copy_cache(source, target, label):
    # Copy file by file as .partial then rename. The manifest is removed first and copied last,
    # so an interrupted copy is never mistaken for a complete cache.
    source, target = Path(source), Path(target)
    target.mkdir(parents=True, exist_ok=True)
    (target / 'manifest.json').unlink(missing_ok=True)
    for name in ('tokenizer.json', 'train.u16', 'valid.u16', 'manifest.json'):
        started = time.time()
        partial = target / (name + '.partial')
        shutil.copyfile(source / name, partial)
        size = partial.stat().st_size
        if size != (source / name).stat().st_size:
            raise CacheError(f'copy of {name} is incomplete')
        os.replace(partial, target / name)
        if size > 2**20:
            print(f'  {label}: {name} {size / 2**30:.2f} GiB in {time.time() - started:.0f}s')

In [ ]:
def hf_token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return None  # anonymous downloads work, with lower rate limits


def parquet_text_batches(path, batch_size=1024):
    import pyarrow.parquet as pq
    for batch in pq.ParquetFile(path).iter_batches(batch_size=batch_size, columns=['text']):
        yield batch.column(0).to_pylist()


def encode_parquet(tokenizer, eos_id, path, out_path, limit=None, label=''):
    tokens = documents = 0
    started = time.time()
    next_report = 100_000_000
    with open(out_path, 'ab') as out:
        for texts in parquet_text_batches(path, batch_size=256 if limit else 1024):
            encodings = tokenizer.encode_batch(texts)  # parallel across CPU cores
            ids = np.fromiter(
                itertools.chain.from_iterable(itertools.chain(e.ids, (eos_id,)) for e in encodings),
                dtype=np.uint16,
            )
            ids.tofile(out)
            tokens += ids.size
            documents += len(texts)
            if tokens >= next_report:
                rate = tokens / (time.time() - started)
                print(f'    {label}: {tokens / 1e6:,.0f}M tokens, {rate / 1e6:.2f}M tokens/s')
                next_report += 100_000_000
            if limit is not None and tokens >= limit:
                break
    return tokens, documents


def build_cache():
    from huggingface_hub import HfApi, hf_hub_download
    from tokenizers import decoders, models, pre_tokenizers, trainers

    build_started = time.time()
    api = HfApi(token=hf_token())
    revision = api.dataset_info(DATASET).sha
    files = sorted(f for f in api.list_repo_files(DATASET, repo_type='dataset', revision=revision)
                   if f.startswith(DATASET_SUBSET + '/') and f.endswith('.parquet'))
    if len(files) <= TRAIN_SHARDS:
        raise CacheError(f'{DATASET_SUBSET} has only {len(files)} files')
    train_files, valid_file = files[:TRAIN_SHARDS], files[-1]
    print(f'{DATASET} {DATASET_SUBSET} at revision {revision[:12]}: {len(files)} files; '
          f'training on {train_files}, validating on {valid_file}')

    download_dir = Path('/content/fineweb_download')
    staging = Path('/content/token_cache_build')
    shutil.rmtree(staging, ignore_errors=True)
    staging.mkdir(parents=True)

    def fetch(name):
        started = time.time()
        path = Path(hf_hub_download(DATASET, name, repo_type='dataset', revision=revision,
                                    local_dir=download_dir, token=hf_token()))
        print(f'  downloaded {name}: {path.stat().st_size / 2**30:.2f} GiB in {time.time() - started:.0f}s')
        return path

    # 1. Tokenizer, trained on the first documents of the first training file.
    first = fetch(train_files[0])
    tokenizer = Tokenizer(models.BPE())
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(vocab_size=VOCAB_SIZE, special_tokens=[EOS_TOKEN], min_frequency=2,
                                  initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), show_progress=False)

    def first_documents():
        count = 0
        for texts in parquet_text_batches(first):
            for text in texts:
                yield text
                count += 1
                if count >= TOKENIZER_TRAIN_DOCS:
                    return

    started = time.time()
    tokenizer.train_from_iterator(first_documents(), trainer=trainer, length=TOKENIZER_TRAIN_DOCS)
    if tokenizer.get_vocab_size() != VOCAB_SIZE:
        raise CacheError(f'tokenizer training stopped at {tokenizer.get_vocab_size()} entries')
    tokenizer_text = tokenizer.to_str()
    (staging / 'tokenizer.json').write_text(tokenizer_text, encoding='utf-8')
    tokenizer = Tokenizer.from_str(tokenizer_text)  # encode with exactly the saved tokenizer
    eos_id = tokenizer.token_to_id(EOS_TOKEN)
    print(f'  trained a {VOCAB_SIZE:,}-entry BPE tokenizer on {TOKENIZER_TRAIN_DOCS:,} documents '
          f'in {time.time() - started:.0f}s')

    # 2. Encode. Each parquet file is deleted after use to keep the VM disk free.
    splits = {'train': {'tokens': 0, 'documents': 0, 'files': train_files},
              'valid': {'tokens': 0, 'documents': 0, 'files': [valid_file]}}
    for name in train_files:
        path = first if name == train_files[0] else fetch(name)
        tokens, documents = encode_parquet(tokenizer, eos_id, path, staging / 'train.u16', label=name)
        splits['train']['tokens'] += tokens
        splits['train']['documents'] += documents
        path.unlink()
        print(f'  encoded {name}: {tokens:,} tokens from {documents:,} documents')
    path = fetch(valid_file)
    tokens, documents = encode_parquet(tokenizer, eos_id, path, staging / 'valid.u16',
                                       limit=VALID_TOKENS, label=valid_file)
    splits['valid'].update(tokens=tokens, documents=documents)
    path.unlink()
    shutil.rmtree(download_dir, ignore_errors=True)
    print(f'  encoded validation: {tokens:,} tokens from {documents:,} documents')

    for split in splits:
        splits[split]['fingerprint'] = fingerprint(staging / f'{split}.u16')
    manifest = {
        'format_version': 1, 'dataset': DATASET, 'subset': DATASET_SUBSET, 'revision': revision,
        'vocab_size': VOCAB_SIZE, 'eos_token': EOS_TOKEN, 'eos_id': eos_id, 'dtype': 'uint16',
        'tokenizer': 'byte-level BPE (Hugging Face tokenizers)', 'tokenizer_train_documents': TOKENIZER_TRAIN_DOCS,
        'tokenizer_sha256': sha256_bytes(tokenizer_text.encode('utf-8')),
        'splits': splits, 'built_utc': time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime()),
        'build_minutes': round((time.time() - build_started) / 60, 1),
    }
    write_json(staging / 'manifest.json', manifest)
    shutil.rmtree(CACHE_LOCAL, ignore_errors=True)
    os.replace(staging, CACHE_LOCAL)  # the VM copy is complete; now the Drive copy
    print(f'Built the cache in {manifest["build_minutes"]} minutes. Saving it to Drive…')
    copy_cache(CACHE_LOCAL, CACHE_DRIVE, 'VM → Drive')


def load_or_build_cache():
    if not REBUILD_CACHE:
        for label, source in (('VM disk', CACHE_LOCAL), ('Drive', CACHE_DRIVE)):
            if not (source / 'manifest.json').is_file():
                continue
            try:
                if source == CACHE_DRIVE:
                    print('Copying the cache from Drive to the VM disk…')
                    copy_cache(CACHE_DRIVE, CACHE_LOCAL, 'Drive → VM')
                print(f'Checking the cache from {label}:')
                return verify_cache(CACHE_LOCAL)
            except (CacheError, OSError, KeyError) as error:
                print(f'The cache from {label} failed the check: {error}')
    if DEVICE.type == 'cuda' and not ALLOW_BUILD_ON_GPU:
        raise StopHere('No good token cache was found. Build it on a CPU runtime first: '
                       'Runtime → Change runtime type → CPU, then Run all. '
                       '(Or set ALLOW_BUILD_ON_GPU = True to build it here.)')
    build_cache()
    print('Checking the new cache:')
    return verify_cache(CACHE_LOCAL)


MANIFEST, TOKENIZER, STREAMS = load_or_build_cache()
TRAIN, VALID = STREAMS['train'], STREAMS['valid']
EOS_ID = MANIFEST['eos_id']
print(f'Cache OK: {len(TRAIN):,} training and {len(VALID):,} validation tokens, memory-mapped from {CACHE_LOCAL}')
if DEVICE.type != 'cuda':
    raise StopHere('The token cache is ready on Drive. Switch to an A100 or L4 runtime '
                   '(Runtime → Change runtime type) and select Run all again.')
require_training_gpu()

## 3. Model: CMU Llama 2 blocks at GPT-2 Small size

The blocks follow CMU 11-711's *Build Your Own Llama* (Llama 2): pre-norm RMSNorm, rotary position embeddings on interleaved pairs, grouped-query attention, and a SwiGLU feed-forward layer, with no biases. Twelve layers of width 768 match GPT-2 Small.

| Part | Parameters |
| --- | ---: |
| Token embedding, 32,000 × 768 | 24,576,000 |
| 12 blocks: attention 1,572,864 + SwiGLU 4,718,592 + two norms 1,536 | 75,515,904 |
| Final norm | 768 |
| Output projection, 768 × 32,000 (separate from the embedding) | 24,576,000 |
| **Total** | **124,668,672** |

Initialization follows GPT-2: normal with standard deviation 0.02, and 0.02 / √(2 × 12) for the two projections that write into the residual stream. Attention uses PyTorch's `scaled_dot_product_attention`, which picks the flash kernel on the A100 and L4. Key/value heads are repeated to match the query heads before the call so the flash kernel always applies.

In [ ]:
@dataclass(frozen=True)
class LlamaConfig:
    vocab_size: int = VOCAB_SIZE
    context: int = CONTEXT
    width: int = 768
    layers: int = 12
    query_heads: int = 12
    kv_heads: int = 4
    ffn_width: int = 2048
    rope_theta: float = 10_000.0
    norm_eps: float = 1e-5


class RMSNorm(nn.Module):
    def __init__(self, width, eps):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(width))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.square().mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight).to(x.dtype)


def rope_tables(head_dim, context, theta):
    inv_freq = 1.0 / theta ** (torch.arange(0, head_dim, 2).float() / head_dim)
    angles = torch.outer(torch.arange(context).float(), inv_freq)
    return angles.cos(), angles.sin()


def apply_rope(x, cos, sin):
    # x: [batch, heads, time, head_dim]; rotates adjacent pairs (0,1), (2,3), … as in Llama 2.
    xf = x.float().unflatten(-1, (-1, 2))
    even, odd = xf[..., 0], xf[..., 1]
    rotated = torch.stack((even * cos - odd * sin, even * sin + odd * cos), dim=-1)
    return rotated.flatten(-2).to(x.dtype)


class Attention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.heads, self.kv_heads = config.query_heads, config.kv_heads
        self.head_dim = config.width // config.query_heads
        self.wq = nn.Linear(config.width, self.heads * self.head_dim, bias=False)
        self.wk = nn.Linear(config.width, self.kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(config.width, self.kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(self.heads * self.head_dim, config.width, bias=False)

    def forward(self, x, cos, sin):
        batch, time_, _ = x.shape
        q = self.wq(x).view(batch, time_, self.heads, self.head_dim).transpose(1, 2)
        k = self.wk(x).view(batch, time_, self.kv_heads, self.head_dim).transpose(1, 2)
        v = self.wv(x).view(batch, time_, self.kv_heads, self.head_dim).transpose(1, 2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        repeat = self.heads // self.kv_heads
        k, v = k.repeat_interleave(repeat, dim=1), v.repeat_interleave(repeat, dim=1)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.wo(y.transpose(1, 2).reshape(batch, time_, -1))


class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.w1 = nn.Linear(config.width, config.ffn_width, bias=False)  # gate
        self.w3 = nn.Linear(config.width, config.ffn_width, bias=False)  # up
        self.w2 = nn.Linear(config.ffn_width, config.width, bias=False)  # down

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attention_norm = RMSNorm(config.width, config.norm_eps)
        self.attention = Attention(config)
        self.ffn_norm = RMSNorm(config.width, config.norm_eps)
        self.feed_forward = FeedForward(config)

    def forward(self, x, cos, sin):
        x = x + self.attention(self.attention_norm(x), cos, sin)
        return x + self.feed_forward(self.ffn_norm(x))


class Llama(nn.Module):
    def __init__(self, config):
        super().__init__()
        if config.width % config.query_heads or config.query_heads % config.kv_heads:
            raise ValueError('width and head counts must divide evenly')
        self.config = config
        self.tok_embeddings = nn.Embedding(config.vocab_size, config.width)
        self.layers = nn.ModuleList(Block(config) for _ in range(config.layers))
        self.norm = RMSNorm(config.width, config.norm_eps)
        self.output = nn.Linear(config.width, config.vocab_size, bias=False)
        cos, sin = rope_tables(config.width // config.query_heads, config.context, config.rope_theta)
        self.register_buffer('rope_cos', cos, persistent=False)
        self.register_buffer('rope_sin', sin, persistent=False)
        residual_std = 0.02 / math.sqrt(2 * config.layers)
        for name, param in self.named_parameters():
            if name.endswith(('wo.weight', 'w2.weight')):
                nn.init.normal_(param, std=residual_std)
            elif param.dim() == 2:
                nn.init.normal_(param, std=0.02)

    def forward(self, ids):
        time_ = ids.shape[1]
        cos, sin = self.rope_cos[:time_], self.rope_sin[:time_]
        x = self.tok_embeddings(ids)
        for layer in self.layers:
            x = layer(x, cos, sin)
        return self.output(self.norm(x))


CONFIG = LlamaConfig()
EXPECTED_PARAMS = 124_668_672
torch.manual_seed(SEED)
RAW_MODEL = Llama(CONFIG).to(DEVICE)
PARAM_COUNT = sum(p.numel() for p in RAW_MODEL.parameters())
assert PARAM_COUNT == EXPECTED_PARAMS, f'{PARAM_COUNT:,} parameters, expected {EXPECTED_PARAMS:,}'
INIT_STATE = {k: v.detach().to('cpu', copy=True) for k, v in RAW_MODEL.state_dict().items()}
if USE_COMPILE:
    import torch._dynamo
    for option in ('cache_size_limit', 'recompile_limit'):
        if hasattr(torch._dynamo.config, option):
            setattr(torch._dynamo.config, option, 64)
MODEL = torch.compile(RAW_MODEL, dynamic=False) if USE_COMPILE else RAW_MODEL

NON_EMBEDDING = PARAM_COUNT - RAW_MODEL.tok_embeddings.weight.numel()
FLOPS_PER_TOKEN = 6 * NON_EMBEDDING + 12 * CONFIG.layers * CONFIG.width * CONTEXT
TOKENS_PER_STEP = EFFECTIVE_BATCH * CONTEXT
print(f'{PARAM_COUNT:,} parameters | {FLOPS_PER_TOKEN / 1e9:.3f} GFLOPs per training token | '
      f'{TOKENS_PER_STEP:,} tokens per step | torch.compile: {USE_COMPILE}')


def reset_model():
    RAW_MODEL.load_state_dict(INIT_STATE)
    RAW_MODEL.train()

## 4. Training utilities

- **Batches.** Each optimizer step draws 512 random windows of 1,025 tokens (input plus the next-token target) from the memory-mapped training stream with a seeded generator. Every learning rate sees exactly the same windows. Gradient accumulation splits the 512 windows into micro-batches, so the effective batch never changes.
- **Precision.** bf16 autocast for the forward and backward passes, with FP32 weights and AdamW state. No gradient scaler is needed.
- **Optimizer.** Fused AdamW with β = (0.9, 0.95) and weight decay 0.1 on matrices only (not on norms), plus gradient clipping at 1.0.
- **Schedule.** Linear warmup over 250 steps, then the constant peak rate: the stable phase of the WSD schedule the final run uses.
- **Validation.** The same 512 windows from `valid.u16` every time, so losses are comparable across rates and steps.
- **Checkpoints.** Written to the VM disk first, copied to Drive as `….partial`, renamed, and only then is the previous checkpoint deleted and purged from the trash. There's at most one training checkpoint on Drive.

In [ ]:
OFFSETS = np.arange(CONTEXT + 1)


def sample_starts(stream, count, generator):
    return torch.randint(len(stream) - CONTEXT - 1, (count,), generator=generator).numpy()


def load_windows(stream, starts):
    windows = torch.from_numpy(np.asarray(stream[starts[:, None] + OFFSETS], dtype=np.int64))
    if DEVICE.type == 'cuda':
        return windows.pin_memory().to(DEVICE, non_blocking=True)
    return windows


VAL_STARTS = sample_starts(VALID, VAL_WINDOWS, torch.Generator().manual_seed(SEED + 1))


def lm_loss(model, ids):
    with torch.autocast(DEVICE.type, dtype=torch.bfloat16):
        logits = model(ids[:, :-1])
    return F.cross_entropy(logits.float().flatten(0, 1), ids[:, 1:].flatten())


@torch.no_grad()
def evaluate(model, microbatch):
    model.eval()
    total = torch.zeros((), device=DEVICE)
    for i in range(0, len(VAL_STARTS), microbatch):
        starts = VAL_STARTS[i:i + microbatch]
        total += lm_loss(model, load_windows(VALID, starts)) * len(starts)
    model.train()
    return total.item() / len(VAL_STARTS)


def make_optimizer(lr):
    decay = [p for p in RAW_MODEL.parameters() if p.dim() >= 2]
    no_decay = [p for p in RAW_MODEL.parameters() if p.dim() < 2]
    return torch.optim.AdamW(
        [{'params': decay, 'weight_decay': WEIGHT_DECAY}, {'params': no_decay, 'weight_decay': 0.0}],
        lr=lr, betas=BETAS, eps=1e-8, fused=DEVICE.type == 'cuda',
    )


def scheduled_lr(step, peak_lr, warmup=WARMUP_STEPS):
    return peak_lr * min(1.0, step / warmup)


def train_step(model, optimizer, generator, microbatch, batch=EFFECTIVE_BATCH):
    # One optimizer step over `batch` windows. Returns (loss, grad_norm, finite).
    starts = sample_starts(TRAIN, batch, generator)
    optimizer.zero_grad(set_to_none=True)
    accumulation = batch // microbatch
    total = torch.zeros((), device=DEVICE)
    for i in range(0, batch, microbatch):
        loss = lm_loss(model, load_windows(TRAIN, starts[i:i + microbatch])) / accumulation
        loss.backward()
        total += loss.detach()
    grad_norm = torch.nn.utils.clip_grad_norm_(RAW_MODEL.parameters(), GRAD_CLIP)
    loss_value, norm_value = torch.stack((total, grad_norm.to(total.dtype))).tolist()
    finite = math.isfinite(loss_value) and math.isfinite(norm_value)
    if finite:
        optimizer.step()
    return loss_value, norm_value, finite


def training_state(optimizer, generator, metrics, spec):
    return {'model': RAW_MODEL.state_dict(), 'optimizer': optimizer.state_dict(),
            'generator': generator.get_state(), 'metrics': metrics, 'spec': spec}


def save_checkpoint(payload, folder, filename, pattern):
    # Save locally, copy to Drive as .partial, rename, then delete every other file matching `pattern`.
    started = time.time()
    folder.mkdir(parents=True, exist_ok=True)
    local = LOCAL_TMP / filename
    torch.save(payload, local)
    partial = folder / (filename + '.partial')
    shutil.copyfile(local, partial)
    if partial.stat().st_size != local.stat().st_size:
        raise OSError(f'incomplete copy of {filename} on Drive')
    os.replace(partial, folder / filename)
    size = local.stat().st_size
    local.unlink()
    for old in folder.glob(pattern):
        if old.name != filename:
            old.unlink()
    for leftover in folder.glob('*.partial'):
        leftover.unlink()
    purged = DRIVE_TRASH.purge()
    print(f'  saved {filename} ({size / 2**30:.2f} GiB) in {time.time() - started:.0f}s'
          + (f'; purged {purged} old file(s) from the Drive trash' if purged else ''))


def load_checkpoint(path):
    local = LOCAL_TMP / path.name
    shutil.copyfile(path, local)
    try:
        return torch.load(local, map_location='cpu', weights_only=False)
    finally:
        local.unlink(missing_ok=True)


def restore(payload, optimizer, generator):
    RAW_MODEL.load_state_dict(payload['model'])
    optimizer.load_state_dict(payload['optimizer'])
    generator.set_state(payload['generator'])


@torch.no_grad()
def sample_text(prompt=SAMPLE_PROMPT, new_tokens=60, temperature=0.8, top_p=0.9, seed=0):
    RAW_MODEL.eval()
    generator = torch.Generator(device=DEVICE).manual_seed(seed)
    ids = torch.tensor([TOKENIZER.encode(prompt).ids], device=DEVICE)
    for _ in range(new_tokens):
        with torch.autocast(DEVICE.type, dtype=torch.bfloat16):
            logits = RAW_MODEL(ids[:, -CONTEXT:])[0, -1].float()
        probs, order = torch.softmax(logits / temperature, dim=-1).sort(descending=True)
        probs[probs.cumsum(0) - probs > top_p] = 0
        choice = order[torch.multinomial(probs / probs.sum(), 1, generator=generator)]
        ids = torch.cat((ids, choice.view(1, 1)), dim=1)
        if choice.item() == EOS_ID:
            break
    RAW_MODEL.train()
    return TOKENIZER.decode(ids[0].tolist())

## 5. Smoke test

A few minutes of checks before spending hours on the sweep. It runs once per GPU type and records the result in `smoke_tests/<gpu>.json` on Drive. It uses the uncompiled model and small batches, so it's quick.

1. The untrained model's validation loss is close to ln(32,000) ≈ 10.37, which is what random guessing gives.
2. The model can **overfit one batch**: the loss on 4 fixed windows falls by more than half in 40 steps. If it can't, something in the model or optimizer is broken.
3. A short training run on fresh batches gives finite losses that go down.
4. **Checkpoint round trip:** save at step 20 through the real Drive path, train 5 more steps, reload the checkpoint, and train the same 5 steps again. The losses must match. This is what makes resuming after a disconnect trustworthy.
5. A second save leaves only one checkpoint in the folder, and the Drive trash doesn't keep the replaced one.

In [ ]:
SMOKE_LR = 1e-3


def run_smoke_test():
    marker = SMOKE_DIR / f'{GPU_TAG}.json'
    if read_json(marker):
        print(f'Smoke test already passed on {GPU_TAG}: {marker}')
        return
    scratch = SMOKE_DIR / 'scratch'
    pattern = f'{FILE_PREFIX}smoke-*.pt'
    results = {'gpu': GPU_NAME, 'torch': torch.__version__}
    quota_before = DRIVE_TRASH.quota()

    # 1. Untrained loss.
    reset_model()
    initial = evaluate(RAW_MODEL, 8)
    results['initial_val_loss'] = initial
    assert abs(initial - math.log(VOCAB_SIZE)) < 0.5, f'initial loss {initial:.3f}, expected ~{math.log(VOCAB_SIZE):.2f}'
    print(f'1. untrained validation loss {initial:.3f} (ln {VOCAB_SIZE:,} = {math.log(VOCAB_SIZE):.3f})')

    # 2. Overfit one batch.
    reset_model()
    optimizer = make_optimizer(SMOKE_LR)
    batch = load_windows(TRAIN, sample_starts(TRAIN, 4, torch.Generator().manual_seed(SEED)))
    losses = []
    for step in range(1, 41):
        for group in optimizer.param_groups:
            group['lr'] = scheduled_lr(step, SMOKE_LR, warmup=5)
        optimizer.zero_grad(set_to_none=True)
        loss = lm_loss(RAW_MODEL, batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(RAW_MODEL.parameters(), GRAD_CLIP)
        optimizer.step()
        losses.append(loss.item())
    results['overfit'] = {'first': losses[0], 'last': losses[-1]}
    assert losses[-1] < 0.5 * losses[0], f'could not overfit one batch: {losses[0]:.3f} → {losses[-1]:.3f}'
    print(f'2. overfit one batch: {losses[0]:.3f} → {losses[-1]:.3f}')

    # 3. Short run on fresh batches, then 4. the checkpoint round trip.
    smoke_batch, smoke_micro = 16, 4
    reset_model()
    optimizer = make_optimizer(SMOKE_LR)
    generator = torch.Generator().manual_seed(SEED + 2)

    def steps(first, last):
        out = []
        for step in range(first, last + 1):
            for group in optimizer.param_groups:
                group['lr'] = scheduled_lr(step, SMOKE_LR, warmup=5)
            loss, _, finite = train_step(RAW_MODEL, optimizer, generator, smoke_micro, batch=smoke_batch)
            assert finite, f'non-finite loss or gradient at smoke step {step}'
            out.append(loss)
        return out

    run = steps(1, 20)
    results['short_run'] = {'first': run[0], 'last_5_mean': float(np.mean(run[-5:]))}
    assert np.mean(run[-5:]) < run[0], 'loss did not go down in 20 steps'
    print(f'3. short run: {run[0]:.3f} → {np.mean(run[-5:]):.3f} (mean of the last 5 steps)')

    save_checkpoint(training_state(optimizer, generator, {'step': 20}, {}), scratch,
                    f'{FILE_PREFIX}smoke-step000020.pt', pattern)
    first_pass = steps(21, 25)
    restore(load_checkpoint(scratch / f'{FILE_PREFIX}smoke-step000020.pt'), optimizer, generator)
    second_pass = steps(21, 25)
    gap = max(abs(a - b) for a, b in zip(first_pass, second_pass))
    results['resume_max_loss_gap'] = gap
    assert gap < 2e-3, f'resumed losses differ by up to {gap:.2e}'
    print(f'4. resume from a Drive checkpoint: the next 5 losses match to {gap:.1e}')

    # 5. One checkpoint at a time, and no growth in the trash.
    save_checkpoint(training_state(optimizer, generator, {'step': 25}, {}), scratch,
                    f'{FILE_PREFIX}smoke-step000025.pt', pattern)
    remaining = sorted(p.name for p in scratch.glob(pattern))
    assert remaining == [f'{FILE_PREFIX}smoke-step000025.pt'], remaining
    for path in scratch.glob(pattern):
        path.unlink()
    time.sleep(5)  # give the Drive mount a moment to move deleted files to the trash
    DRIVE_TRASH.purge()
    quota_after = DRIVE_TRASH.quota()
    if quota_before and quota_after:
        growth = quota_after.get('usageInDriveTrash', 0) - quota_before.get('usageInDriveTrash', 0)
        results['trash_growth_bytes'] = growth
        print(f'5. one checkpoint at a time; Drive trash changed by {growth / 2**20:+.0f} MiB')
    else:
        print('5. one checkpoint at a time (trash not measured: PURGE_DRIVE_TRASH is off). '
              'If Drive usage grows, empty the trash by hand.')

    reset_model()
    del optimizer
    gpu_reset_peak()
    write_json(marker, results)
    print(f'Smoke test passed. Saved {marker}')


if RUN_SMOKE_TEST:
    run_smoke_test()

## 6. Probe: micro-batch, speed and memory

Each candidate micro-batch runs one warm-up step (compiling the model and creating the AdamW state), then timed full optimizer steps of 524,288 tokens, then one validation pass. The fastest micro-batch that leaves at least 1 GiB of GPU memory free is saved in `probe-<gpu>.json`. A later session on the same GPU type reuses it. The probe also turns the measured speed into time estimates for the sweep and for the 2.5B and 10B-token runs.

MFU (model FLOPs utilization) is the measured training FLOPs per second divided by the GPU's peak dense bf16 FLOPs. Expect roughly 30–45% on an A100.

In [ ]:
def hours(tokens, tokens_per_second):
    return tokens / tokens_per_second / 3600


def run_probe(plan_name):
    path = RUN_DIR / f'probe-{GPU_TAG}.json'
    saved = read_json(path)
    if saved and not REPROBE:
        print(f'Using the saved probe for {GPU_TAG}: micro-batch {saved["microbatch"]}, '
              f'{saved["tokens_per_second"]:,.0f} tokens/s')
        return saved
    if not RUN_PROBE:
        raise StopHere(f'No saved probe for {GPU_TAG}. Set RUN_PROBE = True.')
    peak_flops = PEAK_BF16_FLOPS.get(GPU_TAG)
    rows = []
    for microbatch in SWEEP_PLANS[plan_name]['microbatches']:
        if EFFECTIVE_BATCH % microbatch or VAL_WINDOWS % microbatch:
            continue
        reset_model()
        optimizer = make_optimizer(1e-5)
        generator = torch.Generator().manual_seed(SEED)
        gpu_reset_peak()
        free = gpu_free_bytes()
        row = {'microbatch': microbatch}
        try:
            started = time.perf_counter()
            train_step(MODEL, optimizer, generator, microbatch)
            gpu_sync()
            row['first_step_seconds'] = time.perf_counter() - started
            started = time.perf_counter()
            for _ in range(PROBE_TIMED_STEPS):
                train_step(MODEL, optimizer, generator, microbatch)
            gpu_sync()
            row['tokens_per_second'] = PROBE_TIMED_STEPS * TOKENS_PER_STEP / (time.perf_counter() - started)
            started = time.perf_counter()
            evaluate(MODEL, microbatch)
            row['eval_seconds'] = time.perf_counter() - started
        except torch.OutOfMemoryError:
            row['status'] = 'out of memory'
        else:
            peak = gpu_peak_bytes()
            row['peak_gib'] = peak / 2**30
            row['headroom_gib'] = (free - peak) / 2**30
            row['status'] = 'ok' if DEVICE.type != 'cuda' or row['headroom_gib'] >= PROBE_HEADROOM_GIB else 'low headroom'
            if peak_flops:
                row['mfu'] = row['tokens_per_second'] * FLOPS_PER_TOKEN / peak_flops
        del optimizer
        gpu_reset_peak()
        rows.append(row)
        print('  micro-batch {microbatch:>3}: {status:<14} {speed} {peak} {mfu}'.format(
            microbatch=microbatch, status=row['status'],
            speed=f'{row["tokens_per_second"]:>9,.0f} tokens/s' if 'tokens_per_second' in row else '',
            peak=f'| peak {row["peak_gib"]:.1f} GiB, headroom {row["headroom_gib"]:.1f} GiB' if 'peak_gib' in row else '',
            mfu=f'| MFU {row["mfu"]:.0%}' if 'mfu' in row else ''))
    usable = [r for r in rows if r['status'] == 'ok']
    if not usable:
        raise StopHere('No micro-batch fit with enough headroom. Try smaller ones in SWEEP_PLANS.')
    best = max(usable, key=lambda r: r['tokens_per_second'])
    result = {'gpu': GPU_NAME, 'gpu_tag': GPU_TAG, 'torch': torch.__version__, 'rows': rows,
              'microbatch': best['microbatch'], 'tokens_per_second': best['tokens_per_second'],
              'mfu': best.get('mfu'), 'measured_utc': time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime())}
    write_json(path, result)
    reset_model()
    return result


def print_estimates(probe, spec):
    speed = probe['tokens_per_second']
    sweep_tokens = len(spec['learning_rates']) * spec['steps'] * TOKENS_PER_STEP
    print(f'\nMicro-batch {probe["microbatch"]} × {EFFECTIVE_BATCH // probe["microbatch"]} accumulation steps, '
          f'{speed:,.0f} tokens/s on {GPU_TAG}' + (f', MFU {probe["mfu"]:.0%}' if probe.get('mfu') else ''))
    print(f'  this sweep ({len(spec["learning_rates"])} rates × {spec["steps"]} steps, '
          f'{sweep_tokens / 1e9:.2f}B tokens): ~{hours(sweep_tokens, speed):.1f} h of training, plus validation')
    left = sum(spec['steps'] - (read_json(candidate_dir(lr) / 'metrics.json') or {}).get('step', 0)
               for lr in spec['learning_rates']
               if (read_json(candidate_dir(lr) / 'metrics.json') or {}).get('status') not in ('complete', 'diverged'))
    if left < len(spec['learning_rates']) * spec['steps']:
        print(f'  remaining in this sweep: {left:,} steps, ~{hours(left * TOKENS_PER_STEP, speed):.1f} h')
    for tokens in (2.5e9, 10e9):
        print(f'  {tokens / 1e9:g}B tokens on this GPU: ~{hours(tokens, speed):.1f} h')

## 7. Learning-rate sweep

The plan (learning rates and steps) is fixed in `run_config.json` when the experiment starts, from the GPU it starts on. If a later session gets the other GPU, the sweep continues with the same plan and prints the new time estimate.

For each learning rate:

- **Start or resume.** Load the latest checkpoint for this rate if there is one. Otherwise start from the shared initial weights.
- **Train** for the planned steps, logging every 10 steps and running validation every 50 steps and at the end.
- **Checkpoint** every hour, and when `MAX_UPDATES_THIS_SESSION` pauses the run, or when you interrupt the cell.
- **Stop early** if the loss becomes non-finite, or if validation loss after warmup is worse than the untrained model's. The run is then marked `diverged` and the sweep moves on.
- **Finish.** Write `lr-<rate>/metrics.json`, add a short text sample, and delete the checkpoint. If this rate has the best final validation loss so far, its weights are saved in bf16 (about 250 MB) as `best/…best-lr<rate>.pt`, replacing the previous best.

In [ ]:
def plan_spec(plan_name):
    plan = SWEEP_PLANS[plan_name]
    return {
        'format_version': 1, 'plan': plan_name, 'learning_rates': list(plan['learning_rates']),
        'steps': plan['steps'], 'model': asdict(CONFIG), 'parameters': PARAM_COUNT,
        'tokenizer_sha256': MANIFEST['tokenizer_sha256'], 'dataset_revision': MANIFEST['revision'],
        'train_tokens': MANIFEST['splits']['train']['tokens'], 'context': CONTEXT,
        'effective_batch': EFFECTIVE_BATCH, 'tokens_per_step': TOKENS_PER_STEP, 'warmup_steps': WARMUP_STEPS,
        'weight_decay': WEIGHT_DECAY, 'betas': list(BETAS), 'grad_clip': GRAD_CLIP, 'seed': SEED,
        'eval_every': EVAL_EVERY, 'val_windows': VAL_WINDOWS, 'precision': 'bf16 autocast, fp32 master weights',
    }


def load_or_create_spec():
    path = RUN_DIR / 'run_config.json'
    saved = read_json(path)
    if saved is None:
        spec = plan_spec(GPU_PLAN)
        write_json(path, spec)
        print(f'New experiment {EXPERIMENT_NAME}: {GPU_PLAN} plan, rates {spec["learning_rates"]}, '
              f'{spec["steps"]} steps each')
        return spec
    expected = plan_spec(saved['plan'])
    changed = [key for key in expected if expected[key] != saved.get(key)]
    if changed:
        raise StopHere(f'{path} was made with different settings ({", ".join(changed)}). '
                       'Restore those settings, or use a new EXPERIMENT_NAME.')
    if saved['plan'] != GPU_PLAN:
        print(f'Note: this experiment uses the {saved["plan"]} plan; this session has {GPU_TAG}. '
              'Continuing with the saved plan.')
    return saved


def candidate_dir(lr):
    return RUN_DIR / f'lr-{lr:g}'


def checkpoint_name(lr, step):
    return f'{FILE_PREFIX}ckpt-lr{lr:g}-step{step:06d}.pt'


def latest_checkpoint(lr):
    paths = sorted(CKPT_DIR.glob(f'{FILE_PREFIX}ckpt-lr{lr:g}-step*.pt'))
    return paths[-1] if paths else None


def all_results():
    rows = []
    for lr in SPEC['learning_rates']:
        metrics = read_json(candidate_dir(lr) / 'metrics.json')
        if metrics:
            rows.append(metrics)
    return rows


def save_best_weights(metrics):
    finished = [m for m in all_results() if m['status'] == 'complete']
    best = min(finished, key=lambda m: m['final_val_loss'])
    if best['peak_lr'] != metrics['peak_lr']:
        return
    payload = {'config': asdict(CONFIG), 'peak_lr': metrics['peak_lr'], 'step': metrics['step'],
               'final_val_loss': metrics['final_val_loss'], 'tokenizer_sha256': MANIFEST['tokenizer_sha256'],
               'state_dict': {k: v.to(torch.bfloat16) if v.is_floating_point() else v
                              for k, v in RAW_MODEL.state_dict().items()}}
    save_checkpoint(payload, RUN_DIR / 'best', f'{FILE_PREFIX}best-lr{metrics["peak_lr"]:g}.pt',
                    f'{FILE_PREFIX}best-*.pt')
    print(f'  new best: LR {metrics["peak_lr"]:g} (validation loss {metrics["final_val_loss"]:.4f})')


def run_candidate(lr, microbatch, budget):
    # Returns (outcome, updates used). Outcome is 'complete', 'diverged' or 'paused'.
    folder = candidate_dir(lr)
    folder.mkdir(parents=True, exist_ok=True)
    reset_model()
    optimizer = make_optimizer(lr)
    generator = torch.Generator().manual_seed(SEED + 3)  # the same batches for every rate
    checkpoint = latest_checkpoint(lr)
    if checkpoint:
        payload = load_checkpoint(checkpoint)
        if payload['spec'] != SPEC:
            raise StopHere(f'{checkpoint.name} was saved with different settings.')
        restore(payload, optimizer, generator)
        metrics = payload['metrics']
        print(f'LR {lr:g}: resumed from {checkpoint.name}')
    else:
        stale = read_json(folder / 'metrics.json')
        if stale and stale['step'] > 0:
            write_json(folder / f'interrupted-{time.strftime("%Y%m%d-%H%M%S")}.json', stale)
            print(f'LR {lr:g}: no checkpoint for the progress logged in metrics.json; restarting from step 0')
        metrics = {'peak_lr': lr, 'status': 'running', 'step': 0, 'tokens': 0, 'train': [], 'val': [],
                   'max_loss_rise_after_warmup': 0.0, 'min_train_loss': None, 'gpu_tags': []}
        val0 = evaluate(MODEL, microbatch)
        metrics['val'].append({'step': 0, 'tokens': 0, 'val_loss': val0})
        print(f'LR {lr:g}: starting; untrained validation loss {val0:.4f}')
    if GPU_TAG not in metrics['gpu_tags']:
        metrics['gpu_tags'].append(GPU_TAG)
    write_json(folder / 'metrics.json', metrics)

    def checkpoint_now(reason):
        print(f'  checkpoint at step {metrics["step"]} ({reason})')
        save_checkpoint(training_state(optimizer, generator, metrics, SPEC), CKPT_DIR,
                        checkpoint_name(lr, metrics['step']), f'{FILE_PREFIX}ckpt-*.pt')
        write_json(folder / 'metrics.json', metrics)

    used = 0
    last_checkpoint = time.monotonic()
    window_loss, window_steps, window_started = 0.0, 0, time.perf_counter()
    outcome = 'complete'
    step_in_progress, generator_before = None, None
    try:
        for step in range(metrics['step'] + 1, SPEC['steps'] + 1):
            if budget is not None and used >= budget:
                checkpoint_now('session update limit')
                return 'paused', used
            current_lr = scheduled_lr(step, lr)
            for group in optimizer.param_groups:
                group['lr'] = current_lr
            step_in_progress, generator_before = step, generator.get_state()
            loss, grad_norm, finite = train_step(MODEL, optimizer, generator, microbatch)
            step_in_progress = None
            used += 1
            if not finite or loss > 2 * math.log(VOCAB_SIZE):
                print(f'  step {step}: loss {loss}, gradient norm {grad_norm}: diverged')
                metrics['step'] = step
                outcome = 'diverged'
                break
            metrics['step'] = step
            metrics['tokens'] = step * TOKENS_PER_STEP
            window_loss += loss
            window_steps += 1
            if step % LOG_EVERY == 0 or step == SPEC['steps']:
                gpu_sync()
                speed = window_steps * TOKENS_PER_STEP / (time.perf_counter() - window_started)
                mean_loss = window_loss / window_steps
                low = metrics['min_train_loss']
                if step > WARMUP_STEPS and low is not None:
                    metrics['max_loss_rise_after_warmup'] = max(metrics['max_loss_rise_after_warmup'],
                                                                mean_loss - low)
                metrics['min_train_loss'] = mean_loss if low is None else min(low, mean_loss)
                metrics['train'].append({'step': step, 'tokens': metrics['tokens'], 'loss': mean_loss,
                                         'lr': current_lr, 'grad_norm': grad_norm, 'tokens_per_second': speed})
                print(f'  step {step:>4}/{SPEC["steps"]} | loss {mean_loss:.4f} | lr {current_lr:.2e} | '
                      f'grad norm {grad_norm:.2f} | {speed:,.0f} tokens/s')
                window_loss, window_steps, window_started = 0.0, 0, time.perf_counter()
            if step % EVAL_EVERY == 0 or step == SPEC['steps']:
                val = evaluate(MODEL, microbatch)
                metrics['val'].append({'step': step, 'tokens': metrics['tokens'], 'val_loss': val})
                print(f'  step {step:>4} | validation loss {val:.4f}')
                write_json(folder / 'metrics.json', metrics)
                if step > WARMUP_STEPS and val > metrics['val'][0]['val_loss']:
                    print('  validation loss is worse than the untrained model: diverged')
                    outcome = 'diverged'
                    break
                window_started = time.perf_counter()  # don't count validation time in the speed
            if time.monotonic() - last_checkpoint >= CHECKPOINT_EVERY_SECONDS:
                checkpoint_now('hourly')
                last_checkpoint = time.monotonic()
                window_started = time.perf_counter()
    except KeyboardInterrupt:
        if step_in_progress is not None:  # the interrupted step never updated the weights
            generator.set_state(generator_before)
        checkpoint_now('interrupted')
        raise

    metrics['status'] = outcome
    metrics['final_val_loss'] = metrics['val'][-1]['val_loss'] if outcome == 'complete' else None
    if outcome == 'complete':
        metrics['sample'] = {'prompt': SAMPLE_PROMPT, 'text': sample_text()}
        print(f'  sample: {metrics["sample"]["text"]!r}')
    write_json(folder / 'metrics.json', metrics)
    if outcome == 'complete':
        save_best_weights(metrics)
    for path in CKPT_DIR.glob(f'{FILE_PREFIX}ckpt-lr{lr:g}-step*.pt'):
        path.unlink()
    DRIVE_TRASH.purge()
    print(f'LR {lr:g}: {outcome} at step {metrics["step"]}'
          + (f', final validation loss {metrics["final_val_loss"]:.4f}' if metrics['final_val_loss'] else ''))
    return outcome, used


SPEC = load_or_create_spec()
PROBE = run_probe(GPU_PLAN) if (RUN_PROBE or RUN_SWEEP) else None
if PROBE:
    print_estimates(PROBE, SPEC)

In [ ]:
def run_sweep():
    remaining = MAX_UPDATES_THIS_SESSION
    for lr in SPEC['learning_rates']:
        metrics = read_json(candidate_dir(lr) / 'metrics.json')
        if metrics and metrics['status'] in ('complete', 'diverged'):
            print(f'LR {lr:g}: already {metrics["status"]}, skipping')
            continue
        if remaining is not None and remaining <= 0:
            break
        outcome, used = run_candidate(lr, PROBE['microbatch'], remaining)
        if remaining is not None:
            remaining -= used
        gpu_reset_peak()
        if outcome == 'paused':
            print('Paused with a checkpoint on Drive. Run the notebook again to continue.')
            return False
    return all((read_json(candidate_dir(lr) / 'metrics.json') or {}).get('status') in ('complete', 'diverged')
               for lr in SPEC['learning_rates'])


SWEEP_DONE = run_sweep() if RUN_SWEEP else False
print('\nSweep finished.' if SWEEP_DONE else '\nSweep not finished yet.')

## 8. Results

The plot shows validation loss against training tokens for every rate, with training loss as faint lines. The recommendation is the rate with the lowest final validation loss. If a **lower** rate finishes within `CLOSE_MARGIN` (0.01 nats) of it, the lower one is recommended instead, because short runs tend to favor slightly higher rates than a long run can use. Also check the "max rise" column: a large rise in training loss after warmup means spikes, which get worse over a long run.

The summary, plot and per-rate metrics on Drive are small enough to commit to the repo, like Track 1's `runs/` folder.

In [ ]:
def summarize():
    rows = all_results()
    if not rows:
        print('No results yet.')
        return None
    finished = [r for r in rows if r['status'] == 'complete']
    best = min(finished, key=lambda r: r['final_val_loss']) if finished else None
    recommended = best
    if best:
        close = [r for r in finished
                 if r['peak_lr'] < best['peak_lr'] and r['final_val_loss'] - best['final_val_loss'] <= CLOSE_MARGIN]
        if close:
            recommended = min(close, key=lambda r: r['peak_lr'])

    print(f'{"Peak LR":>8} | {"Status":<9} | {"Step":>5} | {"Final val loss":>14} | {"Max rise":>8}')
    for r in rows:
        val = f'{r["final_val_loss"]:.4f}' if r.get('final_val_loss') else '—'
        mark = '  ← recommended' if recommended and r['peak_lr'] == recommended['peak_lr'] else ''
        print(f'{r["peak_lr"]:>8g} | {r["status"]:<9} | {r["step"]:>5} | {val:>14} | '
              f'{r["max_loss_rise_after_warmup"]:>8.3f}{mark}')

    fig, ax = plt.subplots(figsize=(9, 5))
    for i, r in enumerate(rows):
        color = f'C{i}'
        if r['train']:
            ax.plot([p['tokens'] / 1e6 for p in r['train']], [p['loss'] for p in r['train']],
                    color=color, alpha=0.25, linewidth=1)
        ax.plot([p['tokens'] / 1e6 for p in r['val']], [p['val_loss'] for p in r['val']],
                color=color, marker='o', markersize=3, label=f'LR {r["peak_lr"]:g} ({r["status"]})')
    ax.axvline(WARMUP_STEPS * TOKENS_PER_STEP / 1e6, color='grey', linestyle=':', linewidth=1)
    finite_vals = [p['val_loss'] for r in rows for p in r['val'][1:]]
    if finite_vals:
        ax.set_ylim(min(finite_vals) - 0.1, min(max(finite_vals), math.log(VOCAB_SIZE)) + 0.2)
    ax.set(xlabel='Training tokens (millions)', ylabel='Cross-entropy (nats)',
           title='Llama 124M on FineWeb-Edu: learning-rate sweep (dots: validation, lines: training)')
    ax.grid(alpha=0.25)
    ax.legend()
    fig.tight_layout()
    plot_path = RUN_DIR / 'lr_sweep.png'
    fig.savefig(plot_path, dpi=160)
    plt.show()

    summary = {
        'experiment': EXPERIMENT_NAME, 'spec': SPEC, 'probe': PROBE, 'finished': SWEEP_DONE,
        'recommended_peak_lr': recommended['peak_lr'] if recommended else None,
        'lowest_final_val_loss_lr': best['peak_lr'] if best else None,
        'rows': [{key: r.get(key) for key in ('peak_lr', 'status', 'step', 'final_val_loss',
                                               'max_loss_rise_after_warmup', 'gpu_tags', 'sample')}
                 for r in rows],
    }
    write_json(RUN_DIR / 'sweep_summary.json', summary)
    if recommended:
        note = '' if recommended is best else f' (LR {best["peak_lr"]:g} was within {CLOSE_MARGIN} nats)'
        print(f'\nRecommended peak learning rate: {recommended["peak_lr"]:g}{note}')
    print(f'Saved {RUN_DIR / "sweep_summary.json"} and {plot_path}')
    return summary


SUMMARY = summarize()

### Drive layout

```text
MyDrive/CS336_FineWebEdu_Llama124M/
  token_cache/                    # built once on a CPU runtime
    manifest.json  tokenizer.json  train.u16  valid.u16
  smoke_tests/<gpu>.json          # smoke-test results per GPU type
  runs/lr_sweep_v1/
    run_config.json               # the fixed plan and settings
    probe-<gpu>.json              # micro-batch, tokens/s, memory, MFU
    lr-<rate>/metrics.json        # losses, validation curve, text sample
    checkpoints/                  # at most one cs336-llama124m-ckpt-….pt (≈1.5 GB) at a time
    best/                         # bf16 weights of the best rate (≈250 MB)
    sweep_summary.json  lr_sweep.png
```

**Next:** use the recommended rate for the full run, with the same `tokenizer.json`. The full run moved to Colab: see the [Stage 1 notebook](CS336_FineWebEdu_Llama124M_Trained_Stage_1.ipynb) and [why it runs on an L4](../README.md#why-an-l4).

[FineWeb-Edu](https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu) · [Stanford CS336](https://cs336.stanford.edu/) · [CMU 11-711 Build Your Own Llama](https://cmu-l3.github.io/anlp-fall2025/assignments/assignment1) · [PyTorch AMP](https://docs.pytorch.org/docs/stable/amp.html)